In [ ]:
# ======================= 1. Imports =======================
import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import chi2, mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    log_loss,
    matthews_corrcoef,
    cohen_kappa_score
)

from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# ======================= 2. Load CICIDS2017 =======================
print("Loading CICIDS2017 dataset...")

path = "./dataset/"
files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith(".csv")]
dfs = []

for file in files:
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    df['Label'] = df['Label'].apply(
        lambda x: 0 if str(x).upper() == 'BENIGN' else 1
    )

    drop_cols = [
        c for c in df.columns
        if 'Flow ID' in c or 'Timestamp' in c or
           'Source' in c or 'Destination' in c or
           'Protocol' in c or 'Port' in c
    ]
    df.drop(columns=drop_cols, inplace=True, errors='ignore')
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

y = df['Label'].astype(int)
X = df.drop(columns=['Label'])


# ======================= 3. Train / Test Split =======================

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)
# ======================= 4. Preprocessing =======================

imputer = SimpleImputer(strategy="mean")

X_train_full = pd.DataFrame(
    imputer.fit_transform(X_train_full),
    columns=X.columns
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X.columns
)

scaler = MinMaxScaler()

X_train_full = pd.DataFrame(
    scaler.fit_transform(X_train_full),
    columns=X.columns
)

X_test = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns
)

# ======================= 5. AFI-ID Feature Selection =======================

print("\nPerforming Adaptive Feature Intelligence (AFI-ID)...")

mi = mutual_info_classif(
    X_train_full,
    y_train_full,
    random_state=42,
    n_jobs=-1
)

chi = chi2(
    X_train_full,
    y_train_full
)[0]

filter_score = (mi + chi) / 2

xgb_fs = XGBClassifier(
    n_estimators=120,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

et_fs = ExtraTreesClassifier(
    n_estimators=120,
    max_depth=18,
    random_state=42,
    n_jobs=-1
)

xgb_fs.fit(X_train_full, y_train_full)
et_fs.fit(X_train_full, y_train_full)

wrapper_score = (
    xgb_fs.feature_importances_
    + et_fs.feature_importances_
) / 2

final_score = (
    0.4 * filter_score
    + 0.6 * wrapper_score
)

top_k = 28

idx = np.argsort(final_score)[::-1][:top_k]

selected_features = X_train_full.columns[idx]

print(f"Selected {len(selected_features)} features")
print(list(selected_features))

X_train_full = X_train_full[selected_features]
X_test = X_test[selected_features]
# ======================= 6. Train / Validation Split =======================

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    stratify=y_train_full,
    random_state=42
)

X_tr_sm, y_tr_sm = SMOTE(
    random_state=42
).fit_resample(
    X_tr,
    y_tr
)
# ======================= 6. Base Models =======================
lr = LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1)
etc = ExtraTreesClassifier(
    n_estimators=150, max_depth=12,
    random_state=42, n_jobs=-1
)
xgb = XGBClassifier(
    n_estimators=150, max_depth=3,
    learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42, n_jobs=-1
)

lr.fit(X_tr_sm, y_tr_sm)
etc.fit(X_tr_sm, y_tr_sm)
xgb.fit(X_tr_sm, y_tr_sm)

y_pred_lr  = lr.predict(X_test)
y_pred_etc = etc.predict(X_test)
y_pred_xgb = xgb.predict(X_test)


# ======================= 7. Meta-Feature Extraction =======================
def compute_meta_features(model, X):
    p = model.predict_proba(X)
    logits = np.log(p + 1e-8)
    conf = np.max(p, axis=1, keepdims=True)
    ent = -np.sum(p * np.log(p + 1e-8), axis=1, keepdims=True)
    return np.hstack([logits, conf, ent])

train_meta = np.stack([
    compute_meta_features(lr, X_tr_sm),
    compute_meta_features(etc, X_tr_sm),
    compute_meta_features(xgb, X_tr_sm)
], axis=1)

val_meta = np.stack([
    compute_meta_features(lr, X_val),
    compute_meta_features(etc, X_val),
    compute_meta_features(xgb, X_val)
], axis=1)

test_meta = np.stack([
    compute_meta_features(lr, X_test),
    compute_meta_features(etc, X_test),
    compute_meta_features(xgb, X_test)
], axis=1)


# ======================= 8. AFI-ID Attention BiLSTM =======================
class AFI_Attention_BiLSTM(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, 96,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        self.attn = nn.Linear(192, 1)
        self.fc = nn.Sequential(
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        w = F.softmax(self.attn(out).squeeze(-1), dim=1).unsqueeze(-1)
        ctx = torch.sum(out * w, dim=1)
        return self.fc(ctx)


# ======================= 9. Training Setup =======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AFI_Attention_BiLSTM(train_meta.shape[2]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(train_meta, dtype=torch.float32),
        torch.tensor(y_tr_sm.values, dtype=torch.long)
    ),
    batch_size=256, shuffle=True
)

val_loader = DataLoader(
    TensorDataset(
        torch.tensor(val_meta, dtype=torch.float32),
        torch.tensor(y_val.values, dtype=torch.long)
    ),
    batch_size=256
)


# ======================= 10. Training Loop (FORMAT UNCHANGED) =======================
for epoch in range(1, 31):
    model.train()
    ct, tt, tl = 0, 0, 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = criterion(out, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tl += loss.item()
        ct += (out.argmax(1) == yb).sum().item()
        tt += yb.size(0)

    model.eval()
    cv, vl = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            vl += criterion(out, yb).item()
            cv += (out.argmax(1) == yb).sum().item()

    print(
        f"Epoch {epoch:2d}: Train Acc = {ct/tt*100:.2f}%, "
        f"Val Acc = {cv/len(y_val)*100:.2f}%, "
        f"Train Loss = {tl/len(train_loader):.4f}, "
        f"Val Loss = {vl/len(val_loader):.4f}"
    )


# ======================= 11. Final Evaluation =======================
model.eval()
y_pred, y_prob = [], []

with torch.no_grad():
    for xb, _ in DataLoader(
        TensorDataset(
            torch.tensor(test_meta, dtype=torch.float32),
            torch.tensor(y_test.values, dtype=torch.long)
        ),
        batch_size=256
    ):
        xb = xb.to(device)
        out = model(xb)
        y_pred.append(out.argmax(1).cpu().numpy())
        y_prob.append(F.softmax(out, dim=1).cpu().numpy())

y_pred = np.concatenate(y_pred)
y_prob = np.vstack(y_prob)

print("\nAFI-ID Final Metrics")
print("Accuracy:", accuracy_score(y_test, y_pred)*100)
print("Log Loss:", log_loss(y_test, y_prob))
print("MCC:", matthews_corrcoef(y_test, y_pred))
print("Kappa:", cohen_kappa_score(y_test, y_pred))


# ======================= 12. Classification Performance Table =======================
print("\nClassification Performance Table:")
print(f"{'Classifier':<25} {'Class':<12} {'Precision':<10} {'Recall':<10} {'F1 Score':<10} {'Accuracy (%)':<12}")
print("-"*88)

models = {
    "Logistic Regression": y_pred_lr,
    "Extra Trees": y_pred_etc,
    "XGBoost": y_pred_xgb,
    "AFI-ID": y_pred
}

for name, preds in models.items():
    p, r, f, _ = precision_recall_fscore_support(y_test, preds, zero_division=0)
    acc = accuracy_score(y_test, preds)*100
    for i, cls in enumerate(["Normal", "Intrusion"]):
        acc_disp = f"{acc:.2f}" if i == 0 else ""
        print(f"{name:<25} {cls:<12} {p[i]:.4f}    {r[i]:.4f}    {f[i]:.4f}    {acc_disp:<12}")
